# Milestone 3 — Point 4: Feature Selection, Feature Engineering & Encoding

**Project:** Airline Arrival Delay Analysis  
**Dataset:** `flights_2008_clean.csv`  
**Target:** `ArrDelay` (arrival delay in minutes)

This notebook continues from Milestone 2. It does not train a machine-learning model.
It selects predictors that are available before departure, creates useful scheduled-time
features, and defines categorical encoding without fitting preprocessing on the test data.

## 1. Load the cleaned dataset

Place `flights_2008_clean.csv` in the same folder as this notebook.

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "flights_2008_clean.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

## 2. Define the prediction task and eligible rows

We predict arrival delay for flights that actually operated and were not diverted.
Cancelled flights have no meaningful arrival delay; diverted flights have a different
arrival outcome. We therefore exclude them from this specific prediction dataset,
while keeping the original cleaned CSV unchanged.

In [ ]:
analysis_df = df.loc[
    (df["Cancelled"] == 0) &
    (df["Diverted"] == 0) &
    (df["ArrDelay"].notna())
].copy()

print("Rows in cleaned dataset:", len(df))
print("Rows eligible for arrival-delay prediction:", len(analysis_df))
print("Excluded rows:", len(df) - len(analysis_df))

## 3. Feature engineering: scheduled clock times

`CRSDepTime` and `CRSArrTime` are scheduled times stored as HHMM integers.
We convert them to hour and minute components, then create decimal-hour features.
These features are available before departure, unlike actual departure/arrival times.

The helper function validates HHMM values and handles midnight (`0000`).

In [ ]:
def hhmm_to_minutes(series):
    values = pd.to_numeric(series, errors="coerce")
    hours = np.floor(values / 100)
    minutes = values % 100
    valid = values.notna() & hours.between(0, 23) & minutes.between(0, 59)
    result = pd.Series(np.nan, index=series.index, dtype="float64")
    result.loc[valid] = hours.loc[valid] * 60 + minutes.loc[valid]
    return result

for col, prefix in [("CRSDepTime", "ScheduledDep"), ("CRSArrTime", "ScheduledArr")]:
    total_minutes = hhmm_to_minutes(analysis_df[col])
    analysis_df[f"{prefix}Hour"] = total_minutes // 60
    analysis_df[f"{prefix}Minute"] = total_minutes % 60
    analysis_df[f"{prefix}DecimalHour"] = total_minutes / 60

display(analysis_df[[
    "CRSDepTime", "ScheduledDepHour", "ScheduledDepMinute", "ScheduledDepDecimalHour",
    "CRSArrTime", "ScheduledArrHour", "ScheduledArrMinute", "ScheduledArrDecimalHour"
]].head())

### Why these new features?

- Hour and minute components make scheduled clock time easier for algorithms to use.
- Decimal hour represents time on a continuous numeric scale.
- We retain the original scheduled-time columns only as source columns during feature creation;
  the final predictor list uses the engineered versions.

## 4. Feature selection: define the target and predictors

`ArrDelay` is the target. Predictors are restricted to information known before the flight
departs. We exclude actual operational outcomes and delay-cause fields because they are
recorded during/after the flight and would leak information about the answer.

In [ ]:
target_col = "ArrDelay"

numeric_features = [
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "Distance",
    "CRSElapsedTime",
    "ScheduledDepHour",
    "ScheduledDepMinute",
    "ScheduledDepDecimalHour",
    "ScheduledArrHour",
    "ScheduledArrMinute",
    "ScheduledArrDecimalHour",
]

categorical_features = [
    "UniqueCarrier",
    "Origin",
    "Dest",
]

feature_cols = numeric_features + categorical_features

X = analysis_df[feature_cols].copy()
y = analysis_df[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target:", target_col)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 5. Columns deliberately excluded

| Excluded columns | Reason |
|---|---|
| `ArrDelay` from X | It is the target we want to predict. |
| `DepDelay`, `DepTime`, `ArrTime` | Actual operational information; not reliably known before departure. |
| `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay` | Delay causes are recorded after a delay occurs and directly reveal outcome information. |
| `ActualElapsedTime`, `AirTime`, `TaxiIn`, `TaxiOut` | Actual flight-operation measurements unavailable at prediction time. |
| `Cancelled`, `Diverted`, `CancellationCode` | Used to define eligible rows, not as predictors for this task. |
| `FlightNum`, `TailNum` | Identifier-like fields; excluded to avoid memorizing specific flights/aircraft. |
| `Year`, `FlightDate` | Year is constant (2008); date is represented by month/day/weekday features. |
| `ExtremeDelay` | Derived from `ArrDelay`, so using it would directly leak the target. |
| Raw `CRSDepTime`, `CRSArrTime` | Replaced by validated engineered time components. |

## 6. Check missing values in selected features

The selected predictors should be checked before encoding. We do not silently fill missing
values here; imputation belongs in the preprocessing pipeline so it can be fitted using
training data only.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": X.isna().sum(),
    "missing_percent": (X.isna().mean() * 100).round(3)
})
display(missing_summary[missing_summary["missing_count"] > 0])
print("Missing target values:", y.isna().sum())

## 7. Encoding categorical features

`UniqueCarrier`, `Origin`, and `Dest` are nominal categories; their labels do not have a
natural numerical order. We therefore use One-Hot Encoding rather than assigning arbitrary
integer codes.

The encoder is placed inside a scikit-learn pipeline and is **not fitted in this notebook
before the train/test split**. In the next preprocessing step, split the data first, then fit
the encoder on `X_train` only and transform `X_test`. `handle_unknown="ignore"` safely
handles categories appearing only in the test set.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

categorical_encoder = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

categorical_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_encoder, categorical_features)
    ],
    remainder="drop"
)

print("OneHotEncoder configured.")
print("Categorical columns to encode:", categorical_features)
print("Encoder is defined but not fitted yet (prevents data leakage).")

## 8. Final checks and export the selected modeling table

This export contains the selected raw predictors and target. It is not yet the fully encoded
or scaled modeling matrix. The actual encoder fitting and numeric scaling should happen
after the train/test split in the next preprocessing stage.

In [ ]:
assert target_col not in X.columns
assert "DepDelay" not in X.columns
assert "ExtremeDelay" not in X.columns
assert set(categorical_features).issubset(X.columns)
assert set(numeric_features).issubset(X.columns)

print("Selected predictor count:", len(feature_cols))
print("Categorical predictors:", len(categorical_features))
print("Numeric predictors:", len(numeric_features))
print("X and y row counts match:", len(X) == len(y))

modeling_table = X.copy()
modeling_table[target_col] = y
modeling_table.to_csv("milestone_3_selected_features.csv", index=False)

print("Saved: milestone_3_selected_features.csv")

## 9. Summary of decisions

- **Target:** `ArrDelay`.
- **Eligible records:** operated, non-diverted flights with a recorded arrival delay.
- **Engineered features:** scheduled departure and arrival hour/minute/decimal-hour fields.
- **Selected predictors:** calendar features, route distance, scheduled elapsed time, scheduled clock-time features, carrier, origin, and destination.
- **Encoding:** One-Hot Encoding for nominal categorical columns.
- **Leakage prevention:** actual flight outcomes, delay causes, and target-derived columns are excluded; the encoder is defined but not fitted until after splitting.
- **No model is trained** in this milestone.